In [1]:
#models
library(plyr)
library(tidyverse)
library(caret)
library(recipes)
library(pROC)
library(devtools)
library(here)
library(rlucas)
library(reticulate)
#library(gbm)
library(cowplot)
library(FSelectorRcpp) #library(colino)
#library(janitor)
library(dplyr)
library(tidyr)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.1.0     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::arrange()   masks plyr::arrange()
✖ purrr::compact()   masks plyr::compact()
✖ dplyr::count()     masks plyr::count()
✖ dplyr::desc()      masks plyr::desc()
✖ dplyr::failwith()  masks plyr::failwith()
✖ dplyr::filter()    masks stats::filter()
✖ dplyr::id()        masks plyr::id()
✖ dplyr::lag()       masks stats::lag()
✖ dplyr::mutate()    masks plyr::mutate()
✖ dplyr::rename()    masks plyr::rename()
✖ dplyr::summarise() masks plyr::summarise()
✖ dplyr::summarize() masks plyr::summarize()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Loading required package: lattice


Attaching p

### Step 1
#### Load feature dataframe with entire LUCAS-Olink cohort, subset the df to include only patients based on the inclusion criteria

In [2]:
directory_path <- "../../"

In [3]:
# Load the features .csv containing metadata, z_scores, DELFI features, and clinical features
# Add in the multinucratio

features <- read_csv(file.path(directory_path, "data/reproduce_lucas_olink_merged_training_set.csv"), show_col_types=FALSE)
multinucs <- bins5mb %>% group_by(id) %>%
    summarize(multinucratio = sum(multinucs)/(sum(short+long)))
features <- inner_join(multinucs, features, by="id")

###### Criteria for Model 1 #########
#####################################

# Filter out the LUCAS cohort and keep 124 participants with no prior or baseline and 85 patients
# with cancer at the time of the blood draw (n=209)

labels <- meta %>% select(id, assigned_group, Stage)
labels <- labels %>% filter(assigned_group != 1) %>%
    mutate(type = ifelse(assigned_group==2, "healthy", "cancer")) %>%
    select(-assigned_group)

#stage <- meta %>% select(id, Stage)

features <- inner_join(labels, features, by=c("id"="id"))
features <- features  %>% select(-starts_with("cov_"))

features <- features %>% mutate(clinical_smokingstatus=factor(clinical_smokingstatus,
                                                              c("never", "former", "current")),
                                clinical_COPD=as.integer(clinical_COPD))


In [4]:
features_delfi3_final <- read_csv("../../data/delfi3-LUCAS/lucas-features-wide.5mb.hg19.csv",
                                  show_col_types = FALSE)
features_delfi3_final

id,ratio_1,ratio_2,ratio_3,ratio_4,ratio_5,ratio_6,ratio_7,ratio_8,ratio_9,⋯,zscore_17p,zscore_17q,zscore_18p,zscore_18q,zscore_19p,zscore_19q,zscore_20p,zscore_20q,zscore_21q,zscore_22q
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
PGDX26551P1,0.862768797,1.3870584,0.8904268,3.484468,0.423402969,0.46602240,1.8545167,0.173748372,1.00651665,⋯,-0.07668294,-0.7999659,-0.85381987,-1.602845456,0.4591071,1.1332159,-1.570082888,-1.7920345,-0.05249737,-1.67804841
PGDX26758P1,1.348899667,1.3609936,1.0863042,3.493412,0.145277773,0.91021011,1.8484003,0.579129093,0.77967923,⋯,-1.98483866,0.1225240,-0.81660371,-1.209796814,-1.2258007,-1.2777942,-1.185817323,-1.2596899,-1.47405281,0.18485901
PGDX26561P1,-0.300538654,-0.4762504,-0.2030792,3.737315,-0.866604569,0.09105656,1.8659968,0.031727784,0.54098691,⋯,0.03258381,0.8275140,-0.09910527,-0.287687651,-0.3970209,0.1814410,-0.384482422,-0.7774713,0.24203258,-0.08992445
PGDX26721P,1.630617200,1.0286936,1.1017179,3.375796,-0.030057522,0.77887702,2.4217111,0.196652657,1.05802205,⋯,-1.39778755,-2.0817178,-0.83296069,-1.093317004,-1.1090026,-0.4868446,-0.761415647,-1.3950402,-0.97303792,-1.24698570
PGDX26762P1,0.627573850,0.4701624,0.5407863,1.706662,0.148801514,0.32126248,0.9297324,0.002185631,0.04028964,⋯,2.31733491,-0.2058847,0.98853034,-8.305750144,-1.2966500,-0.1866568,3.748289104,5.1878228,-5.14796087,-8.75302265
PGDX26845P,0.132064953,0.1941768,1.0368634,4.423627,-0.979043768,0.32754390,1.4345003,-0.824744663,0.64352102,⋯,-0.46535482,-1.5276119,1.03061099,-0.256769738,-0.5633788,-0.2786615,0.356212713,-1.0812387,-1.96312448,2.20213633
PGDX26615P,1.122759422,1.2046283,1.3736412,2.981961,0.588747614,1.11218533,1.9498378,0.013359518,0.80713976,⋯,-3.91033011,2.8382611,-0.14468900,-2.760847530,-3.4781046,-4.4140127,-3.677498373,-1.0364627,-1.64496117,1.29555181
PGDX26973P,1.228833635,1.4998130,1.3078138,3.223204,0.633817837,0.97897538,1.8882001,0.237645789,0.97875566,⋯,-1.19121453,-2.3813618,-2.67657161,-2.264036980,-0.1915776,0.2466413,-2.678981190,-2.2753932,-1.40822834,-2.50656396
PGDX26633P,0.665947325,0.2810467,0.8652598,3.561201,0.119175310,1.32591034,1.5102479,-0.213558337,0.73948339,⋯,-0.89360346,-1.3840932,0.34366434,-0.182722688,-2.4075143,-0.6836748,0.331567294,0.6665799,-0.18840638,1.58044700


In [5]:
# Identify overlapping columns (excluding 'id')
overlap_cols <- intersect(colnames(features), colnames(features_delfi3_final))
cols_to_drop <- setdiff(overlap_cols, "id")

# Drop overlapping columns from 'features'
features_clean <- features %>% select(-all_of(cols_to_drop))

# Merge, keeping values from features_nova_final
merged_df <- left_join(features_clean, features_delfi3_final, by = "id")


In [6]:
filtered_df <- merged_df %>%
  filter(if_all(starts_with("zscore_") | starts_with("ratio_"), ~ !is.na(.)))

In [7]:
filtered_df

id,Stage,type,multinucratio,clinical_nlratio,clinical_CRP,clinical_cfdna_conc,clinical_age,clinical_IL6,clinical_YKL40,⋯,zscore_17p,zscore_17q,zscore_18p,zscore_18q,zscore_19p,zscore_19q,zscore_20p,zscore_20q,zscore_21q,zscore_22q
<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
PGDX26551P1,NA,healthy,0.07017227,0.6775510,10,NA,96.22450,1.90,236,⋯,-0.07668294,-0.7999659,-0.85381987,-1.602845456,0.45910705,1.13321591,-1.570082888,-1.7920345,-0.05249737,-1.6780484
PGDX26758P1,NA,healthy,0.08047157,0.7447059,10,NA,83.88775,6.00,201,⋯,-1.98483866,0.1225240,-0.81660371,-1.209796814,-1.22580066,-1.27779425,-1.185817323,-1.2596899,-1.47405281,0.1848590
PGDX26721P,NA,healthy,0.05043886,0.7282051,21,10.843182,73.75770,4.00,249,⋯,-1.39778755,-2.0817178,-0.83296069,-1.093317004,-1.10900259,-0.48684455,-0.761415647,-1.3950402,-0.97303792,-1.2469857
PGDX26615P,III,cancer,0.06544904,0.7611765,22,18.936216,60.74743,8.10,96,⋯,-3.91033011,2.8382611,-0.14468900,-2.760847530,-3.47810458,-4.41401268,-3.677498373,-1.0364627,-1.64496117,1.2955518
PGDX26973P,NA,healthy,0.05279855,0.5776316,15,9.758437,68.49829,766.00,101,⋯,-1.19121453,-2.3813618,-2.67657161,-2.264036980,-0.19157761,0.24664127,-2.678981190,-2.2753932,-1.40822834,-2.5065640
PGDX26633P,IV,cancer,0.06607843,0.6026316,10,27.132857,75.92608,2.30,203,⋯,-0.89360346,-1.3840932,0.34366434,-0.182722688,-2.40751432,-0.68367480,0.331567294,0.6665799,-0.18840638,1.5804470
PGDX26648P,IV,cancer,0.08179902,0.8218310,36,48.532500,62.69678,686.00,485,⋯,-28.22856885,2.7444091,34.39590505,29.805959857,-12.22393180,6.77036798,7.653213966,-17.8117107,2.17220401,-37.2859181
PGDX26665P,NA,healthy,0.09654371,0.8021739,10,22.975143,76.48734,4.50,168,⋯,-0.50201972,-1.1722910,-2.52849495,-2.020104065,-0.58311699,0.31722225,-2.038151369,1.3627774,-2.22631539,-2.3348258
PGDX26856P,NA,healthy,0.05936725,0.6767857,10,7.374677,79.87953,3.20,120,⋯,-0.93274702,2.1021700,-0.05630958,0.642854195,-2.58712601,-2.99690003,1.012554271,1.5332114,1.49540454,2.7367207


In [8]:
features <- filtered_df

#### Step 2: Load features dataframe with MICA-Olink cohort, subsetting to patients with Lung cancer and non-cancer (w/o prior cancer)

In [9]:
features_mica <- read_csv("../../data/mica_merged_set.csv", show_col_types=FALSE)
meta_mica <- read_csv("../../data/mica_meta.csv", show_col_types=FALSE)

meta_mica <- meta_mica %>%
  mutate(Stage = ifelse(is.na(Stage), "NA", Stage))

###### Criteria for Model 1 #########
#####################################

# Filter out the MICA cohort and keep participants with no prior, baseline or future cancer

#labels_mica <- meta_mica %>% select(id, Stage, type)
labels_mica <- meta_mica %>% select(id, Stage, assigned_group, 'Coded Type', type, 'CCI score', age, Smoking)

#options(repr.matrix.max.cols = 1000)  # or any high number
#meta_mica

In [10]:
labels_mica <- labels_mica %>%
  mutate(Stage = ifelse(type == "healthy" & Stage != "NA", "NA", Stage))

labels_mica <- labels_mica %>%
  filter(assigned_group == 1 | `Coded Type` == 'Lung (LU)') %>%
  select(-assigned_group)

features_mica <- inner_join(labels_mica, features_mica, by=c("id"="id"))
features_mica <- features_mica  %>% select(-starts_with("cov_"))

# Drop missing CEA values
features_mica <- features_mica %>%
  filter(!is.na(clinical_CEA))

features_mica <- features_mica %>%
  rename(olink_IL6 = olink_IL6_y)

In [11]:
features_mica

id,Stage,Coded Type,type,CCI score,age,Smoking,ratio_1,ratio_2,ratio_3,⋯,olink_LAG3,olink_IL12RB1,olink_IL13,olink_CCL20,olink_TNF,olink_KLRD1,olink_GZMB,olink_CD83,olink_IL12,olink_CSF-1
<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
DL000445NCP0,NA,No cancer (NC),healthy,0,21,0,0.006361435,0.334979190,-0.20862313,⋯,2.33323,1.03951,-0.09898,5.75936,2.36331,6.14441,4.74804,2.59304,6.91326,9.30037
DL000574NCP0,NA,No cancer (NC),healthy,0,40,0,-1.411884503,-1.186855039,-0.60784848,⋯,1.78290,0.82378,0.38476,4.47191,1.50772,5.45484,3.78774,2.53048,6.10674,9.34717
DL000669NCP0,NA,No cancer (NC),healthy,5,68,3,0.534449200,0.706502095,1.01889853,⋯,3.28456,2.58931,1.53026,7.09968,3.83230,6.44849,4.92533,3.55750,7.72471,9.59581
DL000685NCP0,NA,No cancer (NC),healthy,2,63,2,-0.192129239,-0.106697955,0.76730940,⋯,2.67108,1.49650,-0.85737,7.74176,3.15170,6.51259,4.31978,2.61015,6.22409,9.41836
DL000826NCP0,NA,No cancer (NC),healthy,2,64,0,-0.105583668,-1.208594532,-0.48043156,⋯,2.18905,0.87288,-1.09643,5.56785,2.43544,6.77659,4.58297,2.69874,5.74529,9.22055
DL000870NCP0,NA,No cancer (NC),healthy,2,63,0,-0.186834888,0.419620717,1.10495058,⋯,2.05826,0.79965,0.03714,5.65126,2.11777,6.03982,6.19351,3.03127,7.69306,9.87705
DL000379NCP0,NA,No cancer (NC),healthy,4,73,3,-0.393988957,0.105264905,-0.20391870,⋯,2.46383,1.39415,-0.80623,6.16844,2.61852,6.78097,3.91547,2.76802,8.54543,9.30133
DL000245NCP0,NA,No cancer (NC),healthy,3,73,0,-0.332612676,-0.238058138,0.87297666,⋯,2.42538,0.78902,-0.53765,5.26563,2.72932,6.19718,4.09105,2.82034,6.34819,9.26300
DL000353LUP0,II,Lung (LU),cancer,2,67,0,-0.540545083,-1.155989897,0.49063739,⋯,1.97235,1.12715,0.05256,5.04638,2.64812,6.20257,5.26538,2.36902,6.27820,9.20905


### Clean the Olink feature names

In [12]:
colnames(features) <- gsub("olink_IL-1 alpha", "olink_IL-1-alpha", colnames(features))
colnames(features) <- gsub("olink_PDGF subunit B", "olink_PDGF-subunit-B", colnames(features))
colnames(features) <- gsub("olink_LAP TGF-beta-1", "olink_LAP-TGF-beta-1", colnames(features))
colnames(features) <- gsub("-", "_", colnames(features))
colnames(features) <- gsub("/", "_", colnames(features))


colnames(features_mica) <- gsub("olink_IL-1 alpha", "olink_IL-1-alpha", colnames(features_mica))
colnames(features_mica) <- gsub("olink_PDGF subunit B", "olink_PDGF-subunit-B", colnames(features_mica))
colnames(features_mica) <- gsub("olink_LAP TGF-beta-1", "olink_LAP-TGF-beta-1", colnames(features_mica))
colnames(features_mica) <- gsub("-", "_", colnames(features_mica))
colnames(features_mica) <- gsub("/", "_", colnames(features_mica))

### Step 3: Create the MICA calibration and MICA validation sets

In [13]:
set.seed(1234)  # For reproducibility

### MICA calibration set of 40 healthy individuals ###

# Step 1: Randomly select 40 healthy individuals
features_mica_cal <- features_mica %>%
  filter(type == "healthy") %>%
  slice_sample(n = 40)

### Generate MICA validation set ###
# Step 1: Subset all lung cancer individuals
cancer_subset <- features_mica %>%
  filter(type == "cancer")

# Step 2: Randomly select healthy individuals
# BUT exclude the ones used for calibration
healthy_subset <- features_mica %>%
  filter(type == "healthy") %>%
  # Exclude calibration samples by id
  filter(!id %in% features_mica_cal$id)

# Step 3: Combine cancer and filtered healthy individuals for validation
features_mica_validation <- bind_rows(cancer_subset, healthy_subset)

# Optional: Check how many samples were excluded
cat("Original healthy samples:", 
    nrow(features_mica %>% filter(type == "healthy")), "\n")
cat("Healthy samples after excluding calibration set:", nrow(healthy_subset), "\n")
cat("Total validation samples:", nrow(features_mica_validation), "\n")

Original healthy samples: 384 
Healthy samples after excluding calibration set: 344 
Total validation samples: 360 


In [29]:
write_csv(features_mica_validation, "./features_mica_validation.csv")
write_csv(features_mica_cal, "./features_mica_calibration.csv")

### Step 4: Split LUCAS Cohort into sandbox (2/3) and test calibration (1/3)
##### Methodology: (1) Pull out samples in the last two library batches (22 and 23) to include in the test set; (2) add samples to the test set to balance training and testing on (a) Stage, (b) type, and (c) smoking status

In [30]:
### Batch Information ###
batch <- read_csv("../../data/LUCAS_batching.csv", show_col_types=FALSE)

features <- inner_join(features, batch, by="id")



In [31]:
# Set seed for reproducibility
set.seed(1234)

# Ensure 'Stage' is a factor, treating NA as its own category
features$Stage <- as.factor(ifelse(is.na(features$Stage), "NA", as.character(features$Stage)))

# Create a stratification variable combining type, Stage, and clinical_smokingstatus
features$strata <- paste(features$type, features$Stage, features$clinical_smokingstatus, sep = "_")

# Identify samples where library_batch is 22 or 23 and assign them to the test set
batch_test_set <- features[features$library_batch %in% c(22, 23), ]

# Remove these samples from the dataset before stratified sampling
remaining_features <- features[!features$library_batch %in% c(22, 23), ]

# Adjust test proportion to account for batch_test_set
# Calculate the new proportion for stratified sampling
num_batch_test <- nrow(batch_test_set)
num_remaining <- nrow(remaining_features)
total_samples <- nrow(features)

# Adjust the proportion so the final test set remains ~1/3 of total samples
adjusted_test_size <- (total_samples / 3 - num_batch_test) / num_remaining

# Perform stratified sampling on the remaining dataset using adjusted proportion
train_indices <- createDataPartition(remaining_features$strata, p = 1 - adjusted_test_size, list = FALSE)

# Split the remaining dataset
features_train <- remaining_features[train_indices, ]
features_test <- remaining_features[-train_indices, ]

# Combine the test set from stratified sampling with the batch-constrained test set
features_test <- rbind(features_test, batch_test_set)

# Remove the auxiliary column
features_train$strata <- NULL
features_test$strata <- NULL

# Display the number of samples in each group
table(features_train$type, features_train$Stage)
table(features_test$type, features_test$Stage)

# Check dimensions
dim(features_train)
dim(features_test)


Warning message in createDataPartition(remaining_features$strata, p = 1 - adjusted_test_size, :
“Some classes have a single record ( cancer_II_former ) and these will be selected for the sample”


         
           I II III IV NA
  cancer   9  4  18 29  0
  healthy  0  0   0  0 84

         
           I II III IV NA
  cancer   3  2   8 12  0
  healthy  0  0   0  0 40

[1] 144 621

[1]  65 621

In [32]:
#### Drop meta columns from features

# Drop the 'Stage' column from both training and test sets
features_train <- features_train[, !(names(features_train) %in% "Stage")]
features_test <- features_test[, !(names(features_test) %in% "Stage")]
features_train <- features_train[, !(names(features_train) %in% "library_batch")]
features_test <- features_test[, !(names(features_test) %in% "library_batch")]

features <- features[, !(names(features) %in% "Stage")]
features <- features[, !(names(features) %in% "strata")]
features <- features[, !(names(features) %in% "library_batch")]

features_mica <- features_mica %>%
  select(-c(Stage, `CCI score`, `Coded Type`, age, Smoking))

features_mica_cal <- features_mica_cal %>%
  select(-c(Stage, `CCI score`, `Coded Type`, age, Smoking))

features_mica_validation <- features_mica_validation %>%
  select(-c(Stage, `CCI score`, `Coded Type`, age, Smoking))

labels_mica <- labels_mica[, !(names(labels_mica) %in% "CCI score")]
labels_mica <- labels_mica[, !(names(labels_mica) %in% "Coded Type")]
labels_mica <- labels_mica[, !(names(labels_mica) %in% "age")]
labels_mica <- labels_mica[, !(names(labels_mica) %in% "Smoking")]


### Finalize all LUCAS and MICA sets

In [33]:
# Find the common columns
common_cols <- intersect(colnames(features), colnames(features_mica))
length(common_cols)

[1] 602

In [34]:
# Subset both dataframes to the common columns
features_test <- features_test %>% select(all_of(common_cols))
features_train <- features_train %>% select(all_of(common_cols))
features_mica <- features_mica %>% select(all_of(common_cols))
features_mica_cal <- features_mica_cal %>% select(all_of(common_cols))
features_mica_validation <- features_mica_validation %>% select(all_of(common_cols))

In [35]:
options(repr.matrix.max.cols = 15)  # or any high number
features_test

id,type,clinical_CEA,olink_IL8,olink_TNFRSF9,olink_TIE2,olink_MCP_3,olink_CD40_L,⋯,zscore_18q,zscore_19p,zscore_19q,zscore_20p,zscore_20q,zscore_21q,zscore_22q
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
PGDX26551P1,healthy,5,8.66822,6.25843,7.96320,3.46789,9.14967,⋯,-1.60284546,0.45910705,1.1332159,-1.57008289,-1.7920345,-0.05249737,-1.6780484
PGDX26758P1,healthy,5,9.58956,7.03541,7.68914,2.89040,9.13163,⋯,-1.20979681,-1.22580066,-1.2777942,-1.18581732,-1.2596899,-1.47405281,0.1848590
PGDX26822P,healthy,20,10.94263,5.67260,7.98483,3.95008,9.23477,⋯,-0.13969272,-0.61064383,-0.1258775,-0.34567379,-1.0707604,-0.34981276,-1.4900006
PGDX26680P,cancer,297,12.95168,6.90167,8.48563,5.34336,9.88843,⋯,-1.76715407,-0.83025800,3.7368052,3.60749863,4.5420304,-1.86010909,-6.6574084
PGDX26813P1,cancer,5,8.06422,6.34448,7.89992,2.71961,9.81980,⋯,0.07991124,2.02505573,1.5705857,-0.21468547,-1.2117239,0.28315394,-1.1536356
PGDX26924P,healthy,5,7.38962,6.92849,8.19843,3.59275,7.67676,⋯,0.19274100,-0.58166818,0.5709244,1.33128955,-0.7746398,0.83225404,-2.3029072
PGDX26774P,healthy,5,7.24680,6.12795,7.64158,3.43137,9.82715,⋯,-3.58901631,0.31345475,0.4663205,-1.52429626,-0.8766937,-3.40439440,0.4764411
PGDX26696P,cancer,5,10.50197,7.08196,8.36873,6.47135,9.53969,⋯,0.46474449,-1.51319467,-1.0920237,0.53556963,0.7811937,-0.59960571,1.2518912
PGDX26786P,cancer,5,11.99290,6.99933,8.15038,5.43947,9.65662,⋯,-3.02953486,-3.39083456,-5.0824916,-5.19911794,-2.2827373,-2.24373147,0.5856364


In [36]:
features_train

id,type,clinical_CEA,olink_IL8,olink_TNFRSF9,olink_TIE2,olink_MCP_3,olink_CD40_L,⋯,zscore_18q,zscore_19p,zscore_19q,zscore_20p,zscore_20q,zscore_21q,zscore_22q
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
PGDX26721P,healthy,5,7.17222,6.81048,8.39952,3.57847,10.11333,⋯,-1.093317004,-1.10900259,-0.48684455,-0.761415647,-1.39504023,-0.97303792,-1.24698570
PGDX26615P,cancer,10,13.00653,5.85870,7.97369,5.76203,9.86895,⋯,-2.760847530,-3.47810458,-4.41401268,-3.677498373,-1.03646268,-1.64496117,1.29555181
PGDX26633P,cancer,48,10.50250,6.78722,7.97008,5.66763,9.09988,⋯,-0.182722688,-2.40751432,-0.68367480,0.331567294,0.66657988,-0.18840638,1.58044700
PGDX26648P,cancer,20,11.35962,6.86959,8.01933,7.00020,9.94862,⋯,29.805959857,-12.22393180,6.77036798,7.653213966,-17.81171070,2.17220401,-37.28591810
PGDX26665P,healthy,5,8.07218,6.44864,8.47976,3.32627,10.26217,⋯,-2.020104065,-0.58311699,0.31722225,-2.038151369,1.36277741,-2.22631539,-2.33482584
PGDX26856P,healthy,5,7.80417,6.04890,7.16576,2.45356,8.88802,⋯,0.642854195,-2.58712601,-2.99690003,1.012554271,1.53321142,1.49540454,2.73672072
PGDX26664P,cancer,5,10.86086,6.53589,8.43538,5.29850,10.36375,⋯,-0.645375382,-0.65475938,-1.22793519,-2.173055628,-0.38371793,-1.61828265,-0.59392671
PGDX26893P,healthy,5,9.09817,5.84536,7.84820,4.17441,9.42445,⋯,-0.336656833,-2.12579142,-1.63891391,-0.005603114,-0.16341740,0.13216077,1.53123114
PGDX26938P,healthy,5,6.36290,5.75184,7.46164,2.48445,8.23458,⋯,-1.977984764,0.46414577,0.72932052,-0.093228443,-0.93987036,-1.76296763,-2.04564762


In [38]:
write_csv(features_train, "./features_lucas_train.csv")
write_csv(features_test, "./features_lucas_test.csv")
write_csv(features, "./features_lucas_all.csv")

## 